In [ ]:
from utils.analysis.valuation import (
    CompanyAnalyzer,
    ComparisonAnalyzer,
    SectorAnalyzer,
    AnalysisWeights,
    CompanyReporter
)
from utils.tools.config import INVESTMENT_PROFILES

In [ ]:
# 📊 CONFIGURACIÓN DEL ANÁLISIS
# Los pesos determinan la importancia de cada categoría en el score final

# ============================================================================
# SELECCIONA TU PERFIL DE INVERSIÓN
# ============================================================================
# Opciones: 'balanced', 'value', 'growth', 'quality'
# - balanced: Equilibrado en todas las métricas (default)
# - value: Énfasis en valoración (35%) - para value investors
# - growth: Énfasis en crecimiento (45%) - para growth investors
# - quality: Énfasis en rentabilidad y salud (35% cada uno)
# ============================================================================

PROFILE_NAME = 'balanced'  

# Cargar perfil desde config
if PROFILE_NAME and PROFILE_NAME in INVESTMENT_PROFILES:
    profile = INVESTMENT_PROFILES[PROFILE_NAME]
    custom_weights = AnalysisWeights(
        profitability=profile['profitability'],
        financial_health=profile['financial_health'],
        growth=profile['growth'],
        efficiency=profile['efficiency'],
        valuation=profile['valuation']
    )
    print(f"✅ Perfil seleccionado: {PROFILE_NAME.upper()}")
    print(f"   {profile['description']}\n")
else:
    # Usar pesos por defecto (balanced)
    custom_weights = None
    print("✅ Usando pesos por defecto (Balanced)\n")

# Inicializar componentes
analyzer = CompanyAnalyzer(weights=custom_weights)
comparator = ComparisonAnalyzer(company_analyzer=analyzer)
sector_analyzer = SectorAnalyzer(company_analyzer=analyzer)
reporter = CompanyReporter()

# Mostrar configuración actual
print("📋 Pesos configurados:")
print(f"   Rentabilidad:      {analyzer.weights.profitability:.0%}")
print(f"   Salud Financiera:  {analyzer.weights.financial_health:.0%}")
print(f"   Crecimiento:       {analyzer.weights.growth:.0%}")
print(f"   Eficiencia:        {analyzer.weights.efficiency:.0%}")
print(f"   Valoración:        {analyzer.weights.valuation:.0%}")

In [ ]:
# 🔍 ANÁLISIS INDIVIDUAL DE EMPRESA
# Analiza fundamentalmente una empresa usando el perfil configurado

TICKER_TO_ANALYZE = "AAPL" 

print(f"🔍 Analizando {TICKER_TO_ANALYZE} con perfil {PROFILE_NAME.upper()}...\n")

result = analyzer.analyze(TICKER_TO_ANALYZE)

if result.get('success'):
    # Mostrar reporte completo
    print(reporter.render(result))
    
    # Información adicional
    print("\n" + "="*65)
    print("💡 INTERPRETACIÓN:")
    print("="*65)
    conclusion = result['conclusion']
    print(f"Calificación: {conclusion['overall']}")
    print(f"Score Total:  {result['scores']['total']:.1f}/100")
    
    # Mostrar alertas si existen
    all_alerts = []
    for category in ['profitability', 'financial_health', 'growth', 'efficiency', 'valuation']:
        alerts = result.get(category, {}).get('alerts', [])
        all_alerts.extend([(category, alert) for alert in alerts])
    
    if all_alerts:
        print(f"\n⚠️  {len(all_alerts)} ALERTAS DETECTADAS:")
        for cat, alert in all_alerts[:5]:  # Mostrar máximo 5
            print(f"   [{cat}] {alert}")
else:
    print(f"❌ Error: {result.get('error')}")

In [ ]:
# ⚖️  COMPARACIÓN DE MÚLTIPLES EMPRESAS
# Compara fundamentales de varias empresas del mismo sector

TICKERS_TO_COMPARE = ["AAPL", "MSFT", "GOOGL", "META"] 

print(f"⚖️  Comparando {len(TICKERS_TO_COMPARE)} empresas...\n")

comparison = comparator.compare(TICKERS_TO_COMPARE)

if comparison.get('success'):
    # Mostrar tabla resumen
    print("📊 TABLA COMPARATIVA:\n")
    display(comparison['summary_df'])
    
    print(f"\n✅ {comparison['valid_count']}/{len(TICKERS_TO_COMPARE)} empresas analizadas exitosamente")
else:
    print(f"❌ Error en comparación: {comparison.get('error')}")

In [ ]:
# 🏆 RANKING DE EMPRESAS
# Ordena empresas por score total

if comparison.get('success'):
    print("\n🏆 RANKING POR SCORE TOTAL:\n")
    print(f"{'Rank':<6} {'Ticker':<8} {'Empresa':<32} {'Score':<8} {'Conclusión'}")
    print("-" * 70)
    
    for item in comparison['ranking']:
        # Emojis para top 3
        if item['rank'] == 1:
            emoji = "🥇"
        elif item['rank'] == 2:
            emoji = "🥈"
        elif item['rank'] == 3:
            emoji = "🥉"
        else:
            emoji = "  "
        
        name = item['name'][:30] if len(item['name']) > 30 else item['name']
        print(f"{emoji} #{item['rank']:<3} {item['ticker']:<8} {name:<32} "
              f"{item['score']:5.1f}  {item['conclusion']}")

In [ ]:
# 📊 LÍDERES POR CATEGORÍA
# Identifica el mejor y peor en cada dimensión

if comparison.get('success'):
    category_names = {
        'profitability': 'Rentabilidad',
        'financial_health': 'Salud Financiera',
        'growth': 'Crecimiento',
        'efficiency': 'Eficiencia',
        'valuation': 'Valoración'
    }
    
    print("\n📊 LÍDERES POR CATEGORÍA:\n")
    
    for cat, data in comparison['category_leaders'].items():
        name = category_names.get(cat, cat)
        best = data['best']
        worst = data['worst']
        
        print(f"{name}:")
        print(f"  🟢 Mejor:  {best['ticker']:<6} - {best['name'][:25]:<25} (Score: {best['score']:5.1f})")
        print(f"  🔴 Peor:   {worst['ticker']:<6} - {worst['name'][:25]:<25} (Score: {worst['score']:5.1f})")
        print()

In [ ]:
# 📈 ESTADÍSTICAS DEL GRUPO
# Análisis estadístico de las empresas comparadas

if comparison.get('success'):
    stats = comparison['group_stats']
    
    print("\n📈 ESTADÍSTICAS DEL GRUPO:\n")
    
    categories = [
        ('Total', 'total'),
        ('Rentabilidad', 'profitability'),
        ('Salud Financiera', 'financial_health'),
        ('Crecimiento', 'growth'),
        ('Eficiencia', 'efficiency'),
        ('Valoración', 'valuation')
    ]
    
    print(f"{'Categoría':<20} {'Media':<8} {'Mediana':<10} {'Min':<8} {'Max':<8} {'Std':<8}")
    print("-" * 75)
    
    for name, key in categories:
        if key in stats:
            s = stats[key]
            print(f"{name:<20} {s['mean']:6.1f}   {s['median']:7.1f}    "
                  f"{s['min']:6.1f}   {s['max']:6.1f}   {s['std']:6.1f}")

In [ ]:
# 🏭 ANÁLISIS VS PEERS DEL SECTOR
# Compara una empresa específica con competidores del sector

TARGET_COMPANY = "AAPL"  # ← Empresa objetivo
PEERS = ["MSFT", "GOOGL", "META", "AMZN"]  # ← Competidores

print(f"🔬 Analizando {TARGET_COMPANY} vs peers del sector...\n")

sector_result = sector_analyzer.analyze_vs_peers(
    ticker=TARGET_COMPANY,
    peers=PEERS
)

if sector_result.get('success'):
    print(f"EMPRESA: {sector_result['company']['name']} ({TARGET_COMPANY})")
    print(f"Sector: {sector_result['sector']}")
    print(f"Industria: {sector_result['industry']}")
    print(f"Peers analizados: {sector_result['peer_count']}\n")
    
    # Posición relativa
    rel_pos = sector_result['relative_position']
    category_names = {
        'profitability': 'Rentabilidad',
        'financial_health': 'Salud Financiera',
        'growth': 'Crecimiento',
        'efficiency': 'Eficiencia',
        'valuation': 'Valoración',
        'total': 'TOTAL'
    }
    
    print("POSICIÓN RELATIVA:\n")
    
    for cat, data in rel_pos.items():
        if isinstance(data, dict) and 'company_score' in data:
            name = category_names.get(cat, cat.upper())
            diff = data['vs_avg']
            arrow = "↑" if diff > 0 else "↓" if diff < 0 else "="
            
            # Color basado en diferencia
            if diff > 5:
                color = "🟢"
            elif diff < -5:
                color = "🔴"
            else:
                color = "🟡"
            
            print(f"{name}:")
            print(f"  Score empresa:   {data['company_score']:5.1f}")
            print(f"  Promedio peers:  {data['peer_avg']:5.1f}")
            print(f"  {color} Diferencia:    {diff:+5.1f} {arrow}")
            print(f"  Ranking:         #{data['rank']} de {data['total_compared']}")
            print()
    
    # Percentil
    if 'percentiles' in sector_result and 'percentile' in sector_result['percentiles']:
        pct = sector_result['percentiles']
        print(f"📊 PERCENTIL: {pct['percentile']:.1f}%")
        print(f"   Interpretación: {pct['interpretation']}")
        print(f"   Tamaño muestra: {pct['sample_size']} empresas")
else:
    print(f"❌ Error: {sector_result.get('error')}")

In [ ]:
# 🎯 COMPARAR DIFERENTES PERFILES
# Analiza la misma empresa con diferentes estrategias de inversión

TEST_TICKER = "AAPL"  # ← Cambia aquí

print(f"🔬 Analizando {TEST_TICKER} con TODOS los perfiles disponibles:\n")
print(f"{'Perfil':<20} {'Score':<8} {'Conclusión':<15} {'Descripción'}")
print("=" * 80)

for profile_name, profile_config in INVESTMENT_PROFILES.items():
    # Crear analyzer con este perfil
    weights = AnalysisWeights(
        profitability=profile_config['profitability'],
        financial_health=profile_config['financial_health'],
        growth=profile_config['growth'],
        efficiency=profile_config['efficiency'],
        valuation=profile_config['valuation']
    )
    
    profile_analyzer = CompanyAnalyzer(weights=weights)
    result = profile_analyzer.analyze(TEST_TICKER)
    
    if result.get('success'):
        score = result['scores']['total']
        conclusion = result['conclusion']['overall']
        desc = profile_config['description']
        
        print(f"{profile_name.upper():<20} {score:5.1f}    {conclusion:<15} {desc}")
    else:
        print(f"{profile_name.upper():<20} ERROR")

print("\n💡 Observación: El mismo activo puede tener diferente evaluación según tu estrategia")

In [ ]:
# 🏭 ANÁLISIS DE SECTOR COMPLETO
# Analiza un grupo grande de empresas del mismo sector

SECTOR_NAME = "Technology"  # Para referencia
SECTOR_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "META", "NVDA",
    "AMZN", "TSLA", "CRM", "ADBE", "INTC"
]

print(f"🏭 ANÁLISIS DE SECTOR: {SECTOR_NAME}")
print(f"   Analizando {len(SECTOR_TICKERS)} empresas...\n")

sector_comparison = comparator.compare(SECTOR_TICKERS)

if sector_comparison.get('success'):
    ranking = sector_comparison['ranking']
    
    # Top 5
    print("🏆 TOP 5 DEL SECTOR:\n")
    for item in ranking[:5]:
        emoji = "🥇" if item['rank'] == 1 else "🥈" if item['rank'] == 2 else "🥉" if item['rank'] == 3 else "  "
        print(f"{emoji} {item['ticker']:<6} Score: {item['score']:5.1f} - {item['conclusion']}")
    
    # Bottom 5
    print(f"\n⚠️  BOTTOM 5 DEL SECTOR:\n")
    for item in ranking[-5:]:
        print(f"   {item['ticker']:<6} Score: {item['score']:5.1f} - {item['conclusion']}")
    
    # Estadísticas
    scores = [item['score'] for item in ranking]
    print(f"\n📊 ESTADÍSTICAS DEL SECTOR:")
    print(f"   Empresas analizadas: {len(scores)}")
    print(f"   Score promedio:      {sum(scores)/len(scores):.1f}")
    print(f"   Score mediano:       {sorted(scores)[len(scores)//2]:.1f}")
    print(f"   Mejor score:         {max(scores):.1f}")
    print(f"   Peor score:          {min(scores):.1f}")
    print(f"   Desviación estándar: {(sum((x - sum(scores)/len(scores))**2 for x in scores) / len(scores))**0.5:.1f}")
else:
    print(f"❌ Error: {sector_comparison.get('error')}")

In [ ]:
# ℹ️ PERFILES DE INVERSIÓN DISPONIBLES
# Referencia rápida de todos los perfiles configurados

from utils.tools.config import INVESTMENT_PROFILES

print("📋 PERFILES DE INVERSIÓN DISPONIBLES:\n")
print("=" * 80)

for i, (profile_name, config) in enumerate(INVESTMENT_PROFILES.items(), 1):
    print(f"\n{i}️⃣  {profile_name.upper()}")
    print(f"   {config['description']}")
    print(f"   Pesos:")
    print(f"      • Rentabilidad:     {config['profitability']:.0%}")
    print(f"      • Salud Financiera: {config['financial_health']:.0%}")
    print(f"      • Crecimiento:      {config['growth']:.0%}")
    print(f"      • Eficiencia:       {config['efficiency']:.0%}")
    print(f"      • Valoración:       {config['valuation']:.0%}")

print("\n" + "=" * 80)
print("💡 TIP: Cambia PROFILE_NAME en Cell 1 para seleccionar tu estrategia")
print("📚 Los perfiles están definidos en: utils/tools/config.py")